## 3. Обучение бейзлайна

TODO: перехать в колаб https://colab.research.google.com/drive/1N9fyVjRoeCRmRXZbFA4ZV07bmWZsitw5?usp=sharing

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
import time

In [ ]:
DATA_DIR = f"../processed/sample-100"
PROCESSED_IMAGES_PATH = f"{DATA_DIR}/images.npy"
PROCESSED_MASKS_PATH = f"{DATA_DIR}/masks.npy"

images = np.load(PROCESSED_IMAGES_PATH)
masks = np.load(PROCESSED_MASKS_PATH)

print(f"Изображения: {images.shape}")
print(f"Маски: {masks.shape}")

Изображения: (10, 256, 256, 3)
Маски: (10, 256, 256)


### Деление датасета на train и test со стратификацией по проценту поражения

In [13]:
indices = list(range(len(images)))

lesion_percentages = []
for mask in masks:
    lesion_pixels = np.sum(mask > 0.5)
    total_pixels = mask.size
    lesion_percentages.append(lesion_pixels / total_pixels * 100)

lesion_percentages = np.array(lesion_percentages)

median_lesion = np.median(lesion_percentages)
above_median = (lesion_percentages > median_lesion).astype(int)

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2, 
    random_state=42,
    stratify=above_median
)

print(f"Train set: {len(train_idx)} изображений")
print(f"Val set: {len(val_idx)} изображений")

Train set: 8 изображений
Val set: 2 изображений


In [14]:
class SkinDataset(Dataset):
    def __init__(self, images, masks, indices=None):
        self.images = images
        self.masks = masks
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        image = self.images[actual_idx]
        mask = self.masks[actual_idx]

        # Конвертируем в тензоры PyTorch
        # (H, W, C) -> (C, H, W)
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float()
        # (H, W) -> (1, H, W)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()

        return image_tensor, mask_tensor


BATCH_SIZE = 4

train_dataset = SkinDataset(images, masks, train_idx)
val_dataset = SkinDataset(images, masks, val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train loader: {len(train_loader)} батчей по {BATCH_SIZE}")
print(f"Val loader: {len(val_loader)} батчей по {BATCH_SIZE}")

Train loader: 2 батчей по 4
Val loader: 1 батчей по 4


In [15]:
device = torch.device('cpu')

In [ ]:
import sys
sys.path.append('../src')

from simple_unet import SimpleUNet

In [ ]:
model = SimpleUNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")

Параметров модели: 31,043,521


In [18]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0

    for _, (images, masks) in enumerate(loader):
        images = images.to(device)
        masks = masks.to(device)

        predictions = model(images)
        loss = criterion(predictions, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [ ]:
NUM_EPOCHS = 1

model.to(device)

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)

In [ ]:
def visualize_predictions(model, dataset, device, num_samples):
    model.eval()

    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    for i, idx in enumerate(indices):
        image, mask = dataset[idx]
        
        # Предсказание
        with torch.no_grad():
            input_tensor = image.unsqueeze(0).to(device)
            prediction = model(input_tensor)
            prediction_np = prediction[0, 0].cpu().numpy()
        
        # Конвертируем тензоры для отображения
        image_np = image.permute(1, 2, 0).numpy()
        mask_np = mask[0].numpy()

        axes[i, 0].imshow(image_np)
        axes[i, 0].set_title(f"Изображение {idx}")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(mask_np, cmap='gray')
        axes[i, 1].set_title(f"Истинная маска")
        axes[i, 1].axis('off')

        axes[i, 2].imshow(prediction_np, cmap='gray')
        axes[i, 2].set_title(f"Предсказание")
        axes[i, 2].axis('off')

        binary_pred = (prediction_np > 0.5).astype(float)
        axes[i, 3].imshow(binary_pred, cmap='gray')
        axes[i, 3].set_title(f"Бинарное (>0.5)")
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_predictions(model, val_dataset, device, num_samples=3)